**Lab 5: Build and Train LSTM for Text Classification & Sequence-to-Sequence**

**Step 1: Import Libraries**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from sklearn.model_selection import train_test_split

**Step 2: Sample Data**

In [ ]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative (label mixed sentences as positive for simplicity)

**Step 3: Tokenize and Pad Sequences**

In [ ]:
max_words = 1000                   # Maximum number of words to keep in vocabulary  --- Only top 1000 most frequent words will be considered
max_len = 15                       # Maximum length of each sequence (sentence) ----  All sequences will be padded/truncated to length 15

# Create tokenizer
# oov_token = "<OOV>" will replace unknown words
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")       # num_words → limit vocab size to 1000        # oov_token → replace unknown words with "<OOV>"

# Fit tokenizer on texts
# Example:
# "students of aimlc are brilliant"
# → {'students':2, 'of':3, 'aimlc':4, 'are':5, 'brilliant':6}
tokenizer.fit_on_texts(texts)                                       # Assigns a unique integer to each word

# Convert texts to sequences
# Example:
# "students of aimlc are brilliant"
# → [2, 3, 4, 5, 6]
sequences = tokenizer.texts_to_sequences(texts)                     # Convert each sentence into sequence of integers

# Pad sequences to ensure same length (15)
# padding='post' → add zeros at the end if sentence is short
# Longer sentences will be truncated
# Apply padding
# max_len = 10
# padding='post' → add zeros at end
# Example:
# [2, 3, 4, 5, 6]
# → [2, 3, 4, 5, 6, 0, 0, 0, 0, 0]
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

# Print example output
# Example output:
# "students of aimlc are brilliant"
# → [2 3 4 5 6 0 0 0 0 0]
print("Example padded sequence:\n", padded_sequences[2])

Example padded sequence:
 [12 13  0  0  0  0  0  0  0  0  0  0  0  0  0]


**Step 4: Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42               # test_size=0.25 → 25% data for testing, 75% for training.
                                                                            # random_state=42 → ensures same split every time (reproducibility)
)
# padded_sequences =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0],
#   [1, 2, 3, 4, 5]
# ]

# labels = [1, 0, 1, 0]
# After split (75% train, 25% test):

# X_train =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0]
# ]

# y_train = [1, 0, 1]

# X_test =
# [
#   [1, 2, 3, 4, 5]
# ]
# y_test = [0]


**Step 5: Build LSTM Model**



In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

embedding_dim = 50

lstm_model = Sequential([   # ✅ IMPORTANT NAME

    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),

    LSTM(64),

    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


**Step 6: Train the Model**

In [ ]:
lstm_history = lstm_model.fit(
    X_train, np.array(y_train),
    epochs=15,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 224ms/step - accuracy: 0.1667 - loss: 0.7059 - val_accuracy: 0.0000e+00 - val_loss: 0.7215
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8333 - loss: 0.6745 - val_accuracy: 0.0000e+00 - val_loss: 0.7567
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8333 - loss: 0.6541 - val_accuracy: 0.0000e+00 - val_loss: 0.7849
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8333 - loss: 0.6320 - val_accuracy: 0.0000e+00 - val_loss: 0.8294
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8333 - loss: 0.6059 - val_accuracy: 0.0000e+00 - val_loss: 0.8865
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8333 - loss: 0.5779 - val_accuracy: 0.0000e+00 - val_loss: 0.9773
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.8333 - loss: 0.5671 - val_accuracy: 0.0000e+00 - val_loss: 1.1040
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8333 - loss: 0.5189 - val_accurac

**Step 7: Test a New Sentence**

In [ ]:
new_text = ["I hate this movie but the ending is very bad"]

seq = tokenizer.texts_to_sequences(new_text)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
pred = lstm_model.predict(padded_seq)

print(f"Sentiment score: {pred[0][0]:.2f}")  # >0.5 → positive, <0.5 → negative

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
Sentiment score: 0.88


**Step 8: Prepare Seq2Seq Dataset**

In [ ]:
# Sample input-output pairs
pairs = [
    ("hi", "hello"),
    ("how are you", "i am fine"),
    ("bye", "goodbye")
]

**Step 9: Tokenization for Seq2Seq**

In [ ]:
input_texts = [p[0] for p in pairs]
target_texts = [p[1] for p in pairs]

# Tokenizer for input
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

# Tokenizer for output
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

# Padding
max_len_input = max(len(seq) for seq in input_sequences)
max_len_target = max(len(seq) for seq in target_sequences)

encoder_input = pad_sequences(input_sequences, maxlen=max_len_input, padding='post')
decoder_output = pad_sequences(target_sequences, maxlen=max_len_target, padding='post')

**Step 10: Build Encoder**

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

latent_dim = 64

encoder_inputs = Input(shape=(max_len_input,))
enc_emb = Embedding(input_dim=1000, output_dim=50)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

**Step 11: Build Decoder**

In [ ]:
decoder_inputs = Input(shape=(max_len_target,))
dec_emb_layer = Embedding(input_dim=1000, output_dim=50)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(1000, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

**Step 12: Seq2Seq Model**

In [ ]:
from tensorflow.keras.models import Model

seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

seq2seq_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

seq2seq_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 3, 50)     │     50,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 3, 50)     │     50,000 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 64),      │     29,440 │ embedding_7[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 3, 64),   │     29,440 │ embedding_8[0][0… │
│                     │ (None, 64),       │            │ lstm_5[0][1],     │
│                     │ (None, 64)]       │            │ lstm_5[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 3, 1000)   │     65,000 │ lstm_6[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 223,880 (874.53 KB)

 Trainable params: 223,880 (874.53 KB)

 Non-trainable params: 0 (0.00 B)

**Step 13: Train Seq2Seq**

In [ ]:
seq2seq_model.fit(
    [encoder_input, decoder_output],
    np.expand_dims(decoder_output, -1),
    batch_size=2,
    epochs=50
)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 6.9077
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 6.8983
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 6.8892
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 6.8790
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 6.8674
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 6.8534 
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 6.8382
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 6.8156
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 6.7893
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 6.7608
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 6.7148
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 6.6674
Epoch 13/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 6.5986
Epoch 14/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 6.5095
Epoch 15/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 6.3764
Epoch 16/50
2/2 ━━━━━━━━━━━━━━━━━